# Automatic Question Generation: T5 Fine-Tuning Pipeline

This notebook demonstrates the end-to-end process of fine-tuning the `t5-small` model on the SQuAD 1.1 dataset for Question Generation, and then pushing the fine-tuned model directly to the Hugging Face Hub.

In [15]:
# Mount Drive first
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Install Dependencies
First, we install the necessary libraries for model training and dataset handling.

In [16]:
!pip install -q transformers datasets accelerate huggingface_hub torch

## 2. Authenticate with Hugging Face
To push your model to the Hub later, you need to log in using a **Write Token** from your Hugging Face account settings (https://huggingface.co/settings/tokens).

In [17]:
from huggingface_hub import notebook_login

notebook_login()

## 3. Configuration
Define the hyper-parameters and model paths. **Make sure to change `HUB_MODEL_ID` to your own Hugging Face username!**

In [18]:
BASE_MODEL = "t5-small"
#OUTPUT_DIR = "./t5-qg-finetuned"
OUTPUT_DIR = "/content/drive/MyDrive/t5-qg-finetuned"  # ← points to Drive
HUB_MODEL_ID = "Hamzasajjad38/t5-small-qg"  # <-- CHANGE THIS TO YOUR USERNAME

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 64
BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 3e-4
SEED = 42
GRADIENT_ACCUMULATION_STEPS = 2  # ← simulates batch size 8 without OOM

## 4. Load & Prepare the Dataset (SQuAD 1.1)
We stream the SQuAD dataset directly from Hugging Face. We format the input as `generate question: <highlighted_context>` and the target as the actual question.

In [19]:
from datasets import load_dataset

print("Loading SQuAD 1.1 dataset...")
dataset = load_dataset("rajpurkar/squad")  # removed "plain_text" config — not needed

def format_example(example):
    answer = example["answers"]["text"][0]
    context = example["context"]
    question = example["question"]

    # Highlight the answer span in the context
    highlighted = context.replace(answer, f"<hl> {answer} <hl>", 1)
    input_text = f"generate question: {highlighted}"

    return {
        "input_text": input_text,
        "target_text": question,
    }

print("Formatting examples...")
dataset = dataset.map(format_example, remove_columns=dataset["train"].column_names)

# ✂️ Slice to 20k train / 2k validation — fits in one Colab session (~60-75 min)
dataset["train"] = dataset["train"].select(range(20000))
dataset["validation"] = dataset["validation"].select(range(2000))

print(f"✅ Train size: {len(dataset['train'])} | Validation size: {len(dataset['validation'])}")
print("Sample prepared data:", dataset["train"][0])

Loading SQuAD 1.1 dataset...
Formatting examples...
✅ Train size: 20000 | Validation size: 2000
Sample prepared data: {'input_text': 'generate question: Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to <hl> Saint Bernadette Soubirous <hl> in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'target_text': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?'}


## 5. Tokenization
We use the T5 tokenizer to convert our text into numerical input IDs that the model can understand.

In [20]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL)

def tokenize_fn(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        examples["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["input_text", "target_text"])

Tokenizing dataset...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## 6. Model Loading & Training Setup
We load the `t5-small` model and configure the `Seq2SeqTrainer`. This is where we define that the model should automatically be pushed to your Hugging Face Hub repository.

In [21]:
import torch
from transformers import T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    seed=SEED,
    report_to="none",
    push_to_hub=True,                 # This pushes intermediate checkpoints to HF Hub
    hub_model_id=HUB_MODEL_ID,
    gradient_accumulation_steps=2,   # ← add this
    load_best_model_at_end=True,     # ← add this
    generation_max_length=MAX_TARGET_LEN,  # ← add this# Your specific repo name
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    #tokenizer=tokenizer,
    data_collator=data_collator,
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## 7. Fine-Tune the Model
Run the cell below to start the training process. On a free Colab T4 GPU, this will take some time depending on the dataset size and epochs.

In [23]:
print("Starting fine-tuning...")
print(f"Checkpoints saving to: {OUTPUT_DIR}")
trainer.train(resume_from_checkpoint=False) # Changed to False to start training from scratch
print("✅ Training complete!")

Starting fine-tuning...
Checkpoints saving to: /content/drive/MyDrive/t5-qg-finetuned


Epoch,Training Loss,Validation Loss
1,0.862606,0.425869
2,0.784272,0.417912
3,0.676316,0.419243


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


✅ Training complete!


In [3]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=839cace2f16ccbcb333c29cc6cf3b742c048bcfdcf40f4438d17819f6f924f90
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


## 8. Save & Push to Hugging Face
Once training is complete, save the final model and push it to your Hugging Face repository so you can easily use it in your application!

In [24]:
# Save model locally
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved locally to {OUTPUT_DIR}")

# Push final model weights to Hub
print(f"🚀 Pushing model to Hugging Face Hub: {HUB_MODEL_ID} ...")
trainer.push_to_hub(commit_message="Final Fine-tuned T5 for Question Generation")
print(f"✅ Successfully pushed to {HUB_MODEL_ID} on the Hub!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved locally to /content/drive/MyDrive/t5-qg-finetuned
🚀 Pushing model to Hugging Face Hub: Hamzasajjad38/t5-small-qg ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...netuned/training_args.bin: 100%|##########| 5.39kB / 5.39kB            

  ...netuned/model.safetensors:  40%|###9      | 96.0MB /  242MB            

✅ Successfully pushed to Hamzasajjad38/t5-small-qg on the Hub!


# 9-- For evaluation

In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

tokenizer = T5Tokenizer.from_pretrained("Hamzasajjad38/t5-small-qg")
model = T5ForConditionalGeneration.from_pretrained("Hamzasajjad38/t5-small-qg")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"✅ Fine-tuned model loaded from Hub on {device}")

tokenizer_config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

✅ Fine-tuned model loaded from Hub on cuda


In [6]:
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load("en_core_web_sm")

def generate_questions(passage, num_questions=3):
    doc = nlp(passage)
    candidates = list(set([chunk.text for chunk in doc.noun_chunks]))[:num_questions]

    print(f"📝 Passage:\n{passage}\n")
    for answer in candidates:
        highlighted = passage.replace(answer, f"<hl> {answer} <hl>", 1)
        input_text = f"generate question: {highlighted}"
        inputs = tokenizer(input_text, return_tensors="pt",
                           max_length=512, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=64, num_beams=4)
        question = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"  ✅ Answer  : {answer}")
        print(f"  ❓ Question: {question}\n")

# Test it
test_passage = """Photosynthesis is a process used by plants to convert light energy
into chemical energy stored in glucose. It occurs mainly in the chloroplasts
using chlorophyll pigment to absorb sunlight."""

generate_questions(test_passage)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
📝 Passage:
Photosynthesis is a process used by plants to convert light energy
into chemical energy stored in glucose. It occurs mainly in the chloroplasts
using chlorophyll pigment to absorb sunlight.

  ✅ Answer  : It
  ❓ Question: What occurs mainly in the chloroplasts using chlorophyll pigment to absorb sunlight?

  ✅ Answer  : glucose
  ❓ Question: Photosynthesis is a process used by plants to convert light energy into chemical energy stored in what?

  ✅ Answer  : chlorophyll pigment
  ❓ Question: What pigment absorbs sunlight in the chloroplasts?



# For evaluation (ROUGE/BLEU)

In [8]:
from datasets import load_dataset
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu  = evaluate.load("bleu")

# Load only 200 validation samples
val_data = load_dataset("rajpurkar/squad", split="validation[:200]")

predictions, references = [], []

for example in val_data:
    answer  = example["answers"]["text"][0]
    context = example["context"]
    highlighted = context.replace(answer, f"<hl> {answer} <hl>", 1)
    inputs  = tokenizer(f"generate question: {highlighted}",
                        return_tensors="pt", max_length=512,
                        truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=64, num_beams=4)
    pred    = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred)
    references.append(example["question"])

# ROUGE — unchanged
rouge_result = rouge.compute(predictions=predictions,
                             references=references, use_stemmer=True)

# BLEU — fixed: pass plain strings not word lists
bleu_result = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("\n📊 Evaluation Results (200 samples):")
print(f"  ROUGE-1 : {round(rouge_result['rouge1'], 4)}")
print(f"  ROUGE-2 : {round(rouge_result['rouge2'], 4)}")
print(f"  ROUGE-L : {round(rouge_result['rougeL'], 4)}")
print(f"  BLEU    : {round(bleu_result['bleu'],    4)}")


📊 Evaluation Results (200 samples):
  ROUGE-1 : 0.4509
  ROUGE-2 : 0.2445
  ROUGE-L : 0.4167
  BLEU    : 0.1581


In [ ]:
ch